# TF Causal Model — Baselines & Metabolic-GT PR

Compares the **TF→gene GRN** against two ablation baselines on the
**substrate∪product** metabolic-GT task, plus a self-regulon sanity check.
All three scores share the regulon definition (signed RegulonDB mask),
perturbed-gene masking, and coverage mask — they differ *only* in how the
per-(perturbation, TF) value is formed.

| model | mean model | score | ablates |
|---|---|---|---|
| **Linear GRN (TFRegression)** | learned TF→gene map | `|Z_emp|` | — (the method) |
| **ConstantMean** | frozen control mean | `|Z_emp|` | the GRN (apparatus held fixed) |
| **DE (mean |LFC|, raw)** | — (pseudobulk) | mean `|LFC|` over regulon, raw rank | the whole NB/z-score apparatus |

Logic: *A < B < GRN* is the story we want. `B ≈ GRN` ⇒ GRN adds nothing;
`A ≈ GRN` ⇒ apparatus adds nothing.

Training set: **`adata_train` = TF-KDs + control only** (the GRN never sees the
enzyme KDs it is scored on). Scored: held-out non-TF (enzyme) KDs.

---

**§5.1 calibration update.** The per-(perturbation, gene) block `z_pg = R/√V` is *not* `N(0,1)` even on train — fat tails. This is **not** cross-gene covariance (a single gene has none) but an unmodeled **per-perturbation random effect** `b_pg`, amplified ∝ `√n_p` because it is shared by all `n_p` cells of a perturbation while the denominator counts only within-cell NB noise. §4d adds the **between-perturbation variance component** `τ_g²` (DerSimonian–Laird method of moments — no empirical Bayes yet) to the denominator; §5.1 shows it restores calibration. Derivation: `regulon_residual_inference_note_v2.md`.

## 1.0 Imports and data loading

### 1.1 Imports & config

In [ ]:
import sys

sys.path.insert(0, "/workspace/src")

import json
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotnine as gg
import scanpy as sc
from scipy import sparse, stats

import tf_prediction as tfp
from essential.data import load_fitness_data, load_regulondb_full
from essential.utils import PLOTNINE_DEFAULT_THEME_2
from IPython.display import display

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

# Colorblind-friendly map for the matplotlib C-codes used by COLORS below.
CMAP = {"C0": "#4477AA", "C1": "#EE7733", "C2": "#228833", "C3": "#EE6677"}


def show_save(p, name, *, scatter=False, size=(3, 2)):
    """House-themed save (svg, or png for raster/scatter) + inline display."""
    p = p + PLOTNINE_DEFAULT_THEME_2 + gg.theme(figure_size=size)
    p.save(FIGDIR / f"{name}.{'png' if scatter else 'svg'}", dpi=300, verbose=False)
    display(p)
    return p

In [ ]:
ADATA_PATH = "/workspace/data/260309_lce75_genomescale_ezrdm_glu_preprocessed/260309_lce75_genomescale_ezrdm_glu_preprocessed.h5ad"
PERT_COL = "top_target_unthresholded"
CTRL_KEY = "nontargeting"
MIN_LIB = 1e3

N_EPOCHS = 2000
BATCH_SIZE = 512
LR = 1e-3
KEY = jax.random.PRNGKey(0)

MODEL_JSON = "/workspace/data/05142026_metabolome/iML1515.json"
SENSOR_CSV = (
    "/workspace/experiments/06222026_tf/tf_effector_reference/"
    "ledezmatejeida_2022_tf_annotations_resolved.csv"
)
RAPP_LFC_CSV = (
    "/workspace/experiments/06252026_metabolome_transcriptome_data_export/"
    "data/rapp_lfc.csv"
)
DOWNSTREAM_DEPTH = 2  # kept for build_gt_labels shape; we only score sub∪prod
THRESH = 3.0  # |score| cutoff for "discoveries" in the interpretation
PRECISION_KS = [10, 25, 50, 100]

### 1.2 Data & split

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata.obs["_lib"] = np.asarray(adata.layers["reads"].sum(1)).ravel()
adata = adata[(adata.obs["_lib"] > MIN_LIB) & adata.obs[PERT_COL].notna()].copy()
adata.var_names = adata.var_names.str.lower()
adata.obs[PERT_COL] = adata.obs[PERT_COL].str.lower()

tfp.prepare_layers(adata)
adata.X = adata.layers["counts"]
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes")

In [ ]:
ref_db = load_regulondb_full()
ref_db = ref_db[ref_db["ri_type"].str.startswith("TF")].copy()

# Reconcile RegulonDB gene names against adata.var_names using RegulonDB's
# own synonym table (GeneProductAllIdentifiersSet.tsv). Bridges the old
# y-gene names in adata with the newer functional names used in RegulonDB.
syn_rows = tfp.load_synonym_rows()
print("regulator_gene:")
ref_db["regulator_gene"] = tfp.reconcile_names(
    ref_db["regulator_gene"], syn_rows, adata.var_names
)
print()
print("target_gene:")
ref_db["target_gene"] = tfp.reconcile_names(
    ref_db["target_gene"], syn_rows, adata.var_names
)

tf_info = (
    ref_db.assign(TF=ref_db["regulator_gene"])
    .groupby("TF")
    .agg(n_edges=("target_gene", "size"))
    .reset_index()
)
valid_tfs = set(tf_info.query("n_edges >= 2")["TF"])
ref_db = ref_db[ref_db["regulator_gene"].isin(valid_tfs)].copy()

all_tfs = set(ref_db["regulator_gene"].unique())
perts = adata.obs[PERT_COL]
tf_perts = (set(perts.unique()) - {CTRL_KEY}) & all_tfs
non_tf_perts = (set(perts.unique()) - {CTRL_KEY}) - all_tfs

adata_train = adata[perts.isin(tf_perts) | (perts == CTRL_KEY)].copy()
adata_val = adata[perts.isin(non_tf_perts)].copy()

print()
print(
    f"Train: {adata_train.n_obs:,} cells ({len(tf_perts)} TF perturbations + control)"
)
print(f"Val:   {adata_val.n_obs:,} cells ({len(non_tf_perts)} non-TF perturbations)")

In [ ]:
mask = tfp.build_tf_mask(adata.var_names, ref_db)
tf_genes = mask["tf_genes"]
n_genes, n_tfs = adata.n_vars, len(tf_genes)
print(f"{n_tfs} TFs | {int(mask['Amask_tf'].sum()):,} edges")

# ctrl_mask = np.asarray(adata_train.obs[PERT_COL] == CTRL_KEY)
ctrl_mask = np.ones_like(adata_train.obs[PERT_COL], dtype=bool)
std = tfp.TFStandardizer.fit(
    adata_train, tf_cols=mask["tf_cols"], control_mask=ctrl_mask
)
arrays_train = std.transform(adata_train)
arrays_val = std.transform(adata_val)

train_pert_labels = np.asarray(adata_train.obs[PERT_COL])
val_pert_labels = np.asarray(adata_val.obs[PERT_COL])
val_perts = np.unique(val_pert_labels)

### 1.3 Metabolic GT (substrate∪product)

GT axes are the enzymatic subset of `val_perts` × `tf_genes`, identical to the
score axes built in Section 4, so the matrices align row-for-row.

In [ ]:
sensor = pd.read_csv(SENSOR_CSV)
print("sensor Transcription factor:")
sensor["Transcription factor"] = tfp.reconcile_names(
    sensor["Transcription factor"], syn_rows, adata.var_names
)
mm = tfp.MetabolicModel(MODEL_JSON)

enzymatic_idx = np.array([i for i, p in enumerate(val_perts) if mm.has_enzyme(p)])
enzymatic_perts = [val_perts[i] for i in enzymatic_idx]

gt = tfp.build_gt_labels(
    mm,
    sensor,
    perturbations=enzymatic_perts,
    tf_names=list(tf_genes),
    depth=DOWNSTREAM_DEPTH,
)
Y = gt.Y_substrate_or_product  # (n_enz, n_tfs) bool
cov = gt.coverage  # (n_enz, n_tfs) bool — evaluate where True

print()
print(f"enzymatic perturbations: {len(enzymatic_perts)} / {len(val_perts)}")
print(f"  TFs with >=1 BiGG effector: {int(cov.any(0).sum())} / {n_tfs}")

## 2. Inference

One block per model; each produces (i) a **discovery score** matrix on the
enzymatic perts (for Figures 1/1b/3) and (ii) a **self-regulon raw `Z`** vector
on TF-KDs (for Figure 2).

| model | mean model | score | role |
|---|---|---|---|
| **Linear GRN (TFRegressionFixed)** | learned TF→gene map (log link) | `|Z_emp|` | — (the method) |
| **GRN + fitness (TFRegressionFixedNuisance)** | learned TF→gene map + dense fitness block | `|Z_emp|` | asks whether absorbing per-perturbation fitness changes discoveries |
| **ConstantMean** | frozen control mean | `|Z_emp|` | ablates the GRN (apparatus held fixed) |
| **DE (mean |LFC|, raw)** | — (pseudobulk) | mean `|LFC|` over regulon, raw rank | ablates the whole NB/z-score apparatus |

**Shared signed weights (between GRN and ConstantMean).** These two NB scorers use
the *same* signed regulon weights `W_signed = sign(W ⊙ Amask)` taken from the
plain-GRN fit — so ConstantMean differs from the GRN *only* in the predicted
mean (clean ablation). The **nuisance** model uses its **own** `W_signed`
(taken from `W_nuis`), since its edge signs are what a user of that model would
actually apply. The DE baseline ignores sign entirely (mean `|LFC|`), matching
what an analyst would naively do.

In [ ]:
def fit_model(model, X_tf, Y_raw, lib):
    state, hist = tfp.fit(
        model,
        X_tf,
        Y_raw,
        lib,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        key=KEY,
        tqdm_=False,
    )
    return state.params, hist


def zscores_on(model, params, X_tf, Y_raw, lib, pert_labels, W_signed, *, recalibrate):
    mom = tfp.per_perturbation_moments(
        model, params, X_tf, Y_raw, lib, pert_labels, batch_size=BATCH_SIZE
    )
    mom = tfp.mask_perturbed_gene(mom, adata.var_names)
    z = tfp.tf_zscores(
        mom, W_signed=W_signed, Amask_tf=mask["Amask_tf"], recalibrate=recalibrate
    )
    return mom, z


def self_regulon_Z(mom, z):
    # (TF-KD, that TF's OWN regulon): iterate over knocked-down genes; keep the
    # ones that are themselves TFs, and read the score of their own regulon.
    #   z.Z[i, j] = score of perturbation mom.perts[i] on TF tf_genes[j]'s targets
    #   self-regulon entry  <=>  perturbation i IS the TF j  (KD of t, regulon of t)
    tf_col = {t: j for j, t in enumerate(tf_genes)}
    out = []
    for i, p in enumerate(mom.perts):  # p = knocked-down gene
        j = tf_col.get(p)  # is p a TF (does it have a regulon)?
        if j is not None and np.isfinite(z.Z[i, j]):
            out.append(z.Z[i, j])  # KD of TF p, scored on TF p's targets
    return np.array(out)

### 2a. Linear GRN (TFRegression) — the method

In [ ]:
# grn = tfp.TFRegression(
grn = tfp.TFRegressionFixed(
    n_genes=n_genes,
    n_tfs=n_tfs,
    x_mean=jnp.asarray(std.ctrl_lcp_mean),
    Amask_tf=jnp.asarray(mask["Amask_tf"]),
)
params_grn, hist_grn = fit_model(grn, *arrays_train)
print(f"GRN final train NLL: {hist_grn['train_nll'][-1]:.4f}")

W_eff = np.asarray(params_grn["W"]) * mask["Amask_tf"]
W_signed = np.sign(W_eff)  # shared by the GRN and ConstantMean scorers

mom_val_grn, z_val_grn = zscores_on(
    grn, params_grn, *arrays_val, val_pert_labels, W_signed, recalibrate=True
)
mom_tr_grn, z_tr_grn = zscores_on(
    grn, params_grn, *arrays_train, train_pert_labels, W_signed, recalibrate=False
)
assert np.array_equal(mom_val_grn.perts, val_perts)

score_grn = np.abs(z_val_grn.Z_emp)[enzymatic_idx]
self_grn = self_regulon_Z(mom_tr_grn, z_tr_grn)

### 2a-bis. GRN + fitness (TFRegressionFixedNuisance)

Same log-link linear GRN as §2a, plus an additive dense block for a
**per-perturbation fitness covariate** (Calvo-Villamañán 2020 dCas9 log2FC,
`T3`, avg over spacers). The covariate never enters `W` (kept out of the
regulon), so it absorbs fitness-correlated shifts out of the NB residuals
without perturbing the regulon-level z-score. This model uses its **own**
`W_signed` (from its own fitted `W`) — see the §2 header.

In [ ]:
adata.obs

In [ ]:
fitness_df

In [ ]:
fitness_df

In [ ]:
fill_value = fitness_df[["T1", "T2", "T3", "T4"]].mean()
fill_value

In [ ]:
# ---- per-cell fitness covariate (Calvo 2020, T3 only) ------------------------
# Same recipe as tf_metabolic_pr_nuisance.ipynb: T3, per-gene avg over spacers,
# lowercased gene names to match perturbation labels, mean-fill for perts with
# no fitness entry (incl. `nontargeting`). Standardised on TRAIN cells, same
# `(mean, std)` applied to val.
fitness_df = load_fitness_data()
# fitness_df_gene = fitness_df.groupby("gene")[["T1", "T2", "T3", "T4"]].mean()
# fitness_t3 = fitness_df_gene["T1"].copy()
# fitness_t3.index = fitness_t3.index.str.lower()
# fitness_t3 = fitness_t3.groupby(level=0).mean()
# fill_value = float(fitness_t3.mean())
# print(f"{len(fitness_t3):,} genes with fitness  |  fill (mean T3) = {fill_value:.4f}")
# def _fitness_per_cell(ad):
#     p = ad.obs[PERT_COL].astype(str)
#     return p.map(fitness_t3).fillna(fill_value).to_numpy(dtype=np.float32)


# cov_train_raw = _fitness_per_cell(adata_train)
# cov_val_raw = _fitness_per_cell(adata_val)
# for name, ad in [("train", adata_train), ("val", adata_val)]:
#     p = ad.obs[PERT_COL].astype(str)
#     hit = p.isin(fitness_t3.index).mean()
#     print(f"{name}: {hit:.1%} of cells map to a real fitness value")

# cov_mu = float(cov_train_raw.mean())
# cov_sigma = float(cov_train_raw.std())
# cov_sigma = cov_sigma if cov_sigma > 1e-8 else 1.0
# cov_train = ((cov_train_raw - cov_mu) / cov_sigma)[:, None]
# cov_val = ((cov_val_raw - cov_mu) / cov_sigma)[:, None]

# X_aug_train = jnp.concatenate([arrays_train.X_tf, jnp.asarray(cov_train)], axis=1)
# X_aug_val = jnp.concatenate([arrays_val.X_tf, jnp.asarray(cov_val)], axis=1)
# print(f"X_tf {arrays_train.X_tf.shape} -> X_aug {X_aug_train.shape}")


fitness_df_spacer = fitness_df[["T1", "T2", "T3", "T4"]].copy()
SPACER_COL = "top_spacer_unthresholded"
FIT_COLS = ["T1", "T2", "T3", "T4"]

fill_value = fitness_df_spacer[FIT_COLS].mean()  # per-column Series (T1..T4)
print(
    f"{len(fitness_df_spacer):,} spacers with fitness  |  fill (per col) = "
    + ", ".join(f"{c}={v:.3f}" for c, v in fill_value.items())
)


def _fitness_per_cell(ad):
    """(n_cells, 4) fitness matrix; per-column mean-fill for missing spacers."""
    p = ad.obs[SPACER_COL].astype(str)
    return (
        fitness_df_spacer[FIT_COLS]
        .reindex(p)
        .fillna(fill_value)
        .to_numpy(dtype=np.float32)
    )


cov_train_raw = _fitness_per_cell(adata_train)
cov_val_raw = _fitness_per_cell(adata_val)
for name, ad in [("train", adata_train), ("val", adata_val)]:
    p = ad.obs[SPACER_COL].astype(str)
    hit = p.isin(fitness_df_spacer.index).mean()
    print(f"{name}: {hit:.1%} of cells map to a real fitness value")

cov_mu = cov_train_raw.mean(0)
cov_sigma = cov_train_raw.std(0)
cov_train = (cov_train_raw - cov_mu) / cov_sigma
cov_val = (cov_val_raw - cov_mu) / cov_sigma

X_aug_train = jnp.concatenate([arrays_train.X_tf, jnp.asarray(cov_train)], axis=1)
X_aug_val = jnp.concatenate([arrays_val.X_tf, jnp.asarray(cov_val)], axis=1)
print(f"X_tf {arrays_train.X_tf.shape} -> X_aug {X_aug_train.shape}")

In [ ]:
import scanpy as sc

In [ ]:
libs = adata.layers["counts"].sum(1).ravel()
log_lib = np.log10(libs)
plt.hist(log_lib, bins=100)

In [ ]:
adata.X = adata.layers["counts"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
# genes = ["fis", "tufa", "rpsa", "rpos"]
# for g in genes:
#     # y = adata[:, g].layers["lcp10k"].toarray().squeeze()
#     y = adata[:, g].X.toarray().squeeze()
#     fitness = (
#         adata.obs[PERT_COL]
#         .map(fitness_t3)
#         .fillna(fill_value)
#         .to_numpy(dtype=np.float32)
#     )
#     plt.scatter(fitness, y, alpha=0.1)
#     print(f"{g}: {stats.pearsonr(fitness, y)[0]:.3f} (Pearson r)")
#     plt.show()

In [ ]:
adata_train.obs["top_target_unthresholded"]

In [ ]:
grn_nuis = tfp.TFRegressionFixedNuisance(
    n_genes=n_genes,
    n_tfs=n_tfs,
    n_cov=4,
    x_mean=jnp.asarray(std.ctrl_lcp_mean),
    Amask_tf=jnp.asarray(mask["Amask_tf"]),
)
params_grnN, hist_grnN = fit_model(
    grn_nuis, X_aug_train, arrays_train.Y_raw, arrays_train.lib
)
print(f"GRN+fitness final train NLL: {hist_grnN['train_nll'][-1]:.4f}")

# Nuisance model uses its OWN W_signed (its edge signs are what a user of this
# model would apply). ConstantMean/DE still share the plain-GRN W_signed.
W_eff_N = np.asarray(params_grnN["W"]) * mask["Amask_tf"]
W_signed_N = np.sign(W_eff_N)

mom_val_grnN, z_val_grnN = zscores_on(
    grn_nuis,
    params_grnN,
    X_aug_val,
    arrays_val.Y_raw,
    arrays_val.lib,
    val_pert_labels,
    W_signed_N,
    recalibrate=True,
)
mom_tr_grnN, z_tr_grnN = zscores_on(
    grn_nuis,
    params_grnN,
    X_aug_train,
    arrays_train.Y_raw,
    arrays_train.lib,
    train_pert_labels,
    W_signed_N,
    recalibrate=False,
)
assert np.array_equal(mom_val_grnN.perts, val_perts)

In [ ]:
score_grnN = np.abs(z_val_grnN.Z_emp)[enzymatic_idx]
self_grnN = self_regulon_Z(mom_tr_grnN, z_tr_grnN)

In [ ]:
W_eff_N.shape

In [ ]:
np.where(mask["Amask_tf"].sum(1) != 0)[0][:20]

In [ ]:
W_pick = W_eff_N[15]
print("non-zero W")
print(W_pick[W_pick != 0])

In [ ]:
adata.var_names[15]

In [ ]:
# ref_db.loc[lambda df: (df["regulator_gene"] == "nagc")]

In [ ]:
adata_train[adata_train.obs[PERT_COL] == "fadr", "acca"].layers[
    "lcp10k"
].toarray()

In [ ]:
reg_gene_name = "nagc"
target_gene_name = "naga"

# reg_gene = adata_train[:, reg_gene_name].layers["lcp10k"].toarray().squeeze()
# target_gene = adata_train[:, target_gene_name].layers["lcp10k"].toarray().squeeze()
reg_gene = adata_train[:, reg_gene_name].layers["counts"].toarray().squeeze()
target_gene = adata_train[:, target_gene_name].layers["counts"].toarray().squeeze()


reg_gene_kd = (
    adata_train[adata_train.obs[PERT_COL] == reg_gene_name, reg_gene_name]
    .layers["lcp10k"]
    .toarray()
)
target_gene_kd = (
    adata_train[adata_train.obs[PERT_COL] == reg_gene_name, target_gene_name]
    .layers["lcp10k"]
    .toarray()
)

plt.scatter(reg_gene, target_gene, alpha=0.1)
plt.scatter(reg_gene_kd, target_gene_kd, color="red")
print(f"Pearson r = {stats.pearsonr(reg_gene, target_gene)[0]:.3f}")
plt.xlabel(f"{reg_gene_name} (TF)")
plt.ylabel(f"{target_gene_name} (target)")

In [ ]:
params_grnN["b"][15]

In [ ]:
plt.hist(np.array(params_grnN["W"])[:, -4:].ravel(), bins=100)

In [ ]:
plt.plot(hist_grn["train_nll"])
plt.plot(hist_grnN["train_nll"])

### 2b. ConstantMean — baseline B (ablates the GRN)

In [ ]:
cm = tfp.ConstantMean(n_genes=n_genes, x_mean=jnp.asarray(std.ctrl_lcp_mean))
params_cm, hist_cm = fit_model(cm, *arrays_train)  # fits per-gene overdispersion only

mom_val_cm, z_val_cm = zscores_on(
    cm, params_cm, *arrays_val, val_pert_labels, W_signed, recalibrate=True
)
mom_tr_cm, z_tr_cm = zscores_on(
    cm, params_cm, *arrays_train, train_pert_labels, W_signed, recalibrate=False
)

score_cm = np.abs(z_val_cm.Z_emp)[enzymatic_idx]
self_cm = self_regulon_Z(mom_tr_cm, z_tr_cm)

### 2c. DE (mean |LFC|, raw) — baseline A (ablates the apparatus)

Pseudobulk mean log-CP10K per perturbation; LFC vs the control (nontargeting)
mean; per (p, TF) score = mean `|LFC|` over the regulon, **directly-KD'd gene
excluded**, **raw rank (no recalibration)** — the closest thing to what one
would do by hand.

In [ ]:
lcp = adata_val.layers["lcp10k"]
lcp = lcp.toarray() if sparse.issparse(lcp) else np.asarray(lcp)
uq, inv = np.unique(val_pert_labels, return_inverse=True)
assert np.array_equal(uq, val_perts)

sums = np.zeros((len(uq), n_genes))
counts = np.zeros(len(uq))
np.add.at(sums, inv, lcp)
np.add.at(counts, inv, 1)
mean_lcp = sums / counts[:, None]
LFC = mean_lcp - std.ctrl_lcp_mean[None, :]  # control = nontargeting train-ctrl mean

ALFC = np.abs(LFC)
gene_idx = {g: i for i, g in enumerate(adata.var_names)}
cols = np.array([gene_idx.get(p, -1) for p in uq])
valid = cols >= 0
ALFC[np.where(valid)[0], cols[valid]] = 0.0  # exclude directly-KD'd gene

num = ALFC @ mask["Amask_tf"]  # (n_perts, n_tfs) Σ|LFC| over regulon
n_targets = mask["Amask_tf"].sum(0)
pir = np.zeros((len(uq), n_tfs))
pir[valid] = mask["Amask_tf"][cols[valid], :]  # is KD'd gene in TF t's regulon?
denom = n_targets[None, :] - pir
denom = np.where(denom > 0, denom, np.nan)
score_de = (num / denom)[enzymatic_idx]
print(
    "DE score matrix:",
    score_de.shape,
    "| finite frac:",
    float(np.isfinite(score_de).mean()),
)

In [ ]:
MODELS = {
    "Linear GRN (TFRegression)": score_grn,
    "GRN + fitness (TFRegressionFixedNuisance)": score_grnN,
    "ConstantMean": score_cm,
    "DE (mean |LFC|, raw)": score_de,
}
SELF = {
    "Linear GRN (TFRegression)": self_grn,
    "GRN + fitness (TFRegressionFixedNuisance)": self_grnN,
    "ConstantMean": self_cm,
}
COLORS = {
    "Linear GRN (TFRegression)": "C0",
    "GRN + fitness (TFRegressionFixedNuisance)": "C1",
    "ConstantMean": "C3",
    "DE (mean |LFC|, raw)": "C2",
}

## 3. Study of residuals

### 3.1 Gene-level z-scores on train data — is the normality assumption met?

The regulon score is a `w`-weighted combination of the **per-(perturbation, gene)**
building block `z_pg = R_pg / sqrt(V_pg)`, which is `~N(0,1)` under the model
(correct conditional mean + cell independence). Checking it directly on the
training data separates two failure modes:

- if `z_pg` is already non-normal here → the per-gene NB studentization itself is
  miscalibrated, and the regulon score merely inherits it;
- if `z_pg ≈ N(0,1)` → the per-gene block is fine, and any departure at the
  regulon level comes from the **aggregation** (cross-gene covariance), not the
  studentization.

#### 3.1b — the same block, calibrated by the variance component

Overlay the variance-component `z_pg` (§4d) on the naive one, on the **same** train moments. The only change is the added `τ_g²` in the denominator (`β_g ≈ 0` in-sample, so the numerator is effectively unchanged). If the diagnosis holds, the naive fat tails collapse onto the `N(0,1)` diagonal, and the removed inflation `√(1 + τ_g²/s_pg²)` is sizable.

In [ ]:
# Naive and variance-component gene-level z on the SAME train moments (GRN).
# Recompute zg here so the cell is self-contained (it is also defined in 5.1 above).
Rnum, Vsum = mom_tr_grn.Rnum, mom_tr_grn.Vsum
zg = np.where(Vsum > 0, Rnum / np.sqrt(np.where(Vsum > 0, Vsum, 1.0)), np.nan).ravel()
zg = zg[np.isfinite(zg)]


bins = np.linspace(-6, 6, 80)
hist_groups = [(zg, CMAP["C0"], "naive  R/√V")]
# hist_groups.append((zc, CMAP["C1"], "variance-component"))
_h = pd.concat(
    pd.DataFrame(
        {
            "z": np.clip(z_, bins[0], bins[-1]),
            "kind": f"{lab}  (sd={z_.std():.2f}, |z|>3: {(np.abs(z_) > 3).mean():.3f})",
        }
    )
    for z_, c_, lab in hist_groups
)
_fill = {
    f"{lab}  (sd={z_.std():.2f}, |z|>3: {(np.abs(z_) > 3).mean():.3f})": c_
    for z_, c_, lab in hist_groups
}
xs = np.linspace(-6, 6, 400)
_norm = pd.DataFrame({"x": xs, "d": np.exp(-(xs**2) / 2) / np.sqrt(2 * np.pi)})
p = (
    gg.ggplot()
    + gg.geom_histogram(
        gg.aes("z", y=gg.after_stat("density"), fill="kind"),
        data=_h,
        bins=80,
        alpha=0.5,
        position="identity",
        color="none",
    )
    + gg.geom_line(gg.aes("x", "d"), data=_norm, linetype="dashed", size=0.5)
    + gg.scale_fill_manual(values=_fill, name="")
    + gg.labs(
        x="per-(perturbation, gene) z (train)",
        y="density",
        subtitle="naive vs variance-component",
    )
    + gg.theme(
        legend_position="bottom",
    )
)
show_save(p, "fig_5_1_genez_hist", size=(2.5, 2))

# (2) QQ point plot vs N(0,1) with y=x reference.
q = np.linspace(0.001, 0.999, 400)
tq = stats.norm.ppf(q)
_qq = pd.DataFrame({"theo": tq, "emp": np.quantile(zg, q)})
lo, hi = stats.norm.ppf([0.001, 0.999])
p = (
    gg.ggplot(_qq, gg.aes("theo", "emp"))
    + gg.geom_point(size=0.6, color=CMAP["C0"])
    # + gg.geom_segment(x=lo, xend=hi, y=lo, yend=hi, linetype="dashed", size=0.5)
    + gg.geom_abline(slope=1, intercept=0, linetype="dashed", size=0.5)
    + gg.labs(
        x="theoretical N(0,1) quantile",
        y="empirical quantile",
        subtitle="QQ — calibration restored",
    )
)
show_save(p, "fig_5_1_genez_qq", scatter=True, size=(2.5, 2))


def _summ(z):
    return (
        f"sd={z.std():6.2f}  min={z.min():8.1f}  max={z.max():8.1f}  "
        f"|z|>3: {(np.abs(z) > 3).mean():.3f}  |z|>6: {(np.abs(z) > 6).mean():.4f}"
    )


print("naive  R/√V        :", _summ(zg))

#### 3.1c — locate the tail: low predicted mean (variance floor) or `n_p` (random effect)?

`min ≈ −5.7` but `max ≈ 5e5` — the tail is **one-sided**, which a (symmetric) per-perturbation random effect cannot produce. Since `z_pg = R/√V` with `V = μ + μ²/φ`, a tiny predicted mean `μ` (floored at `eps=1e-8` in `lcp_to_count_mean`) collapses `V→0` and makes any observed count explode `z`. This cell tests that directly: extreme `|z|` should concentrate at **low predicted mean**, the tail should be **positive**, and the `|z|>3` rate should rise across the predicted-mean axis but stay flat across `n_p`.

In [ ]:
from scipy import stats

# Naive per-(perturbation, gene) z and a predicted-mean proxy on the same moments.
n_p_col = mom_tr_grn.counts.astype(float)[:, None]
Rnum, Vsum = mom_tr_grn.Rnum, mom_tr_grn.Vsum
with np.errstate(divide="ignore", invalid="ignore"):
    Zpg = np.where(Vsum > 0, Rnum / np.sqrt(Vsum), np.nan)  # naive z
    mu_cell = np.where(
        Vsum > 0, Vsum / n_p_col, np.nan
    )  # Vsum/n_p ≈ μ/cell at low expr
np_grid = np.broadcast_to(n_p_col, Zpg.shape)

ok = np.isfinite(Zpg) & np.isfinite(mu_cell)
z, mu, npc = Zpg[ok], mu_cell[ok], np_grid[ok]
extreme = np.abs(z) > 6

# (1) hexbin: |z| vs predicted mean (both log); extremes should sit at low μ.
_hb = pd.DataFrame(
    {
        "x": np.log10(np.clip(mu, 1e-9, None)),
        "y": np.log10(np.clip(np.abs(z), 1e-3, None)),
    }
)
p = (
    gg.ggplot(_hb, gg.aes("x", "y"))
    + gg.geom_bin2d(bins=60)
    + gg.scale_fill_cmap(cmap_name="viridis", trans="log10", name="count")
    + gg.geom_hline(
        yintercept=np.log10(6), color=CMAP["C3"], linetype="dashed", size=0.5
    )
    + gg.geom_vline(xintercept=-8, color="black", linetype="dotted", size=0.5)
    + gg.labs(
        x="log10 predicted mean/cell ≈ Vsum/n_p",
        y="log10 |z|",
        # subtitle="extreme |z| concentrate at low predicted mean",
    )
)
show_save(p, "fig_5_1c_lowmean_hexbin", scatter=True, size=(2.5, 2.0))

# (2) signed QQ on symlog y — exposes the one-sided tail.
q = np.linspace(0.001, 0.999, 600)
_qq = pd.DataFrame({"theo": stats.norm.ppf(q), "emp": np.quantile(z, q)})
lo, hi = stats.norm.ppf([0.001, 0.999])
p = (
    gg.ggplot(_qq, gg.aes("theo", "emp"))
    + gg.geom_point(size=0.6, color=CMAP["C0"])
    + gg.geom_segment(x=lo, xend=hi, y=lo, yend=hi, linetype="dashed", size=0.5)
    + gg.scale_y_continuous(trans="symlog", minor_breaks=[])
    + gg.labs(
        x="theoretical N(0,1) quantile",
        y="empirical z quantile (symlog)",
        # subtitle="QQ — positive tail explodes, negative ~ normal",
    )
)
show_save(p, "fig_5_1c_qq", scatter=True, size=(2.5, 2.0))

# (3) the numbers
print(f"|z|>6 mass:                  {extreme.mean():.4f}  (n={extreme.sum():,})")
print(f"  of those, positive:        {(z[extreme] > 0).mean():.3f}")
print(f"  of those, Vsum/n_p < 1e-6: {(mu[extreme] < 1e-6).mean():.3f}")
print(f"median Vsum/n_p overall:     {np.median(mu):.3g}")
print(f"median Vsum/n_p among |z|>6: {np.median(mu[extreme]):.3g}")
print()


def rate_by_quartile(x, label):
    edges = np.quantile(x, [0, 0.25, 0.5, 0.75, 1.0])
    print(f"|z|>6 rate by {label} quartile:")
    for a, b in zip(edges[:-1], edges[1:]):
        m = (x >= a) & (x <= b)
        if m.any():
            print(
                f"   [{a:10.3g}, {b:10.3g}]  rate={(np.abs(z[m]) > 6).mean():.4f}  n={m.sum():,}"
            )


rate_by_quartile(mu, "predicted-mean")  # expect: rises sharply toward low μ
rate_by_quartile(npc, "n_p")  # expect: ~flat if random effect is not the driver

#### 3.1d — long table: which (perturbation, gene) pairs carry the large z?

Per-(perturbation, gene) table from `mom_tr_grn`, sorted by `|z|`. Two separate expression columns disentangle the candidate causes:

- `mu_cell` = model's predicted mean/cell **for this perturbation** (`Vsum/n_p`); low ⇒ this *(p,g)* is predicted low.
- `ctrl_lcp` = the gene's **baseline** log-CP10K; low ⇒ a genuinely **low-count gene**.

If the tail is only low-count genes, the top rows all have low `ctrl_lcp`. If instead they have **high `ctrl_lcp` but low `mu_cell`**, the model is over-predicting repression (a mean bias), which is a different disease.

In [ ]:
import pandas as pd
from IPython.display import display

# Per-(perturbation, gene) long table from the train moments (GRN), sorted by |z|.
Rnum, Vsum = mom_tr_grn.Rnum, mom_tr_grn.Vsum
n_p = mom_tr_grn.counts.astype(float)
genes = np.asarray(adata.var_names)
perts = np.asarray(mom_tr_grn.perts)
P, G = np.meshgrid(np.arange(len(perts)), np.arange(len(genes)), indexing="ij")

with np.errstate(divide="ignore", invalid="ignore"):
    Z = np.where(Vsum > 0, Rnum / np.sqrt(Vsum), np.nan)
    mu_cell = np.where(
        Vsum > 0, Vsum / n_p[:, None], np.nan
    )  # predicted mean/cell (this pert)

dfz = pd.DataFrame(
    {
        "perturbation": perts[P.ravel()],
        "gene": genes[G.ravel()],
        "z": Z.ravel(),
        "Rnum": Rnum.ravel(),
        "Vsum": Vsum.ravel(),
        "n_p": n_p[P.ravel()],
        "mu_cell": mu_cell.ravel(),  # predicted mean/cell (perturbation-specific)
        "resid_cell": (Rnum / n_p[:, None]).ravel(),  # mean residual/cell = Rnum/n_p
        "ctrl_lcp": std.ctrl_lcp_mean[
            G.ravel()
        ],  # gene BASELINE expr (log-CP10K) -> low-count gene?
        "conc": mom_tr_grn.conc[G.ravel()],
    }
)
dfz = dfz[np.isfinite(dfz["z"])].copy()
dfz["abs_z"] = dfz["z"].abs()
dfz["obs_cell_approx"] = (
    dfz["mu_cell"] + dfz["resid_cell"]
)  # ~ observed count/cell (exact at low mu)
dfz = dfz.sort_values("abs_z", ascending=False).reset_index(drop=True)

# --- does the |z|>6 tail concentrate in low-count GENES, or in over-repressed (p,g)? ---
top = dfz[dfz.abs_z > 6]
med_base = dfz.ctrl_lcp.median()
hi_base_lo_pred = (top.ctrl_lcp > med_base) & (top.mu_cell < 1.0)
print(f"rows: {len(dfz):,}   |z|>6: {len(top):,}")
print(
    f"distinct genes in |z|>6: {top.gene.nunique()} / {dfz.gene.nunique()}   "
    f"perts: {top.perturbation.nunique()} / {dfz.perturbation.nunique()}"
)
print(
    f"ctrl_lcp (baseline) q[.1,.5,.9]  -- |z|>6: {np.round(top.ctrl_lcp.quantile([.1,.5,.9]).values,2)}"
    f"   all: {np.round(dfz.ctrl_lcp.quantile([.1,.5,.9]).values,2)}"
)
print(
    f"mu_cell q[.1,.5,.9]              -- |z|>6: {np.round(top.mu_cell.quantile([.1,.5,.9]).values,3)}"
    f"   all: {np.round(dfz.mu_cell.quantile([.1,.5,.9]).values,3)}"
)
print(
    f"frac of |z|>6 that are HIGH-baseline (>median) but predicted-low (mu_cell<1): {hi_base_lo_pred.mean():.3f}"
)
print("\ntop genes by count of |z|>6 entries:")
print(top.gene.value_counts().head(15))

display(dfz.head(30))

#### 3.1e — naive z, restricted to relevant perturbations

The GRN predicts gene `g` only from `g`'s **regulator TFs**. A KD of an *unrelated* TF barely moves `g`'s predicted mean, yet can induce global/secondary shifts that surface as miscalibrated residuals not attributable to the model's predictive target. Here we keep, **for each gene `g`**, only **controls + KDs of `g`'s known regulator TFs** (`Amask_tf[g, ·]`) — the (p,g) pairs that are actually the model's job — and re-draw the naive `z = R/√V` histogram + QQ on that in-domain subset.

In [ ]:
from scipy import stats

# Per gene g, keep controls + KDs of g's KNOWN regulator TFs (Amask_tf[g, ·]).
perts = np.asarray(mom_tr_grn.perts)
Amask = np.asarray(mask["Amask_tf"])  # (n_genes, n_tfs)
tf_col = {t: j for j, t in enumerate(mask["tf_genes"])}

keep = np.zeros(mom_tr_grn.Rnum.shape, dtype=bool)  # (n_perts, n_genes)
for i, p in enumerate(perts):
    if p == CTRL_KEY:
        keep[i, :] = True  # controls: relevant to every gene
    elif p in tf_col:
        keep[i, :] = Amask[:, tf_col[p]] > 0  # KD of TF p: only p's targets

Rnum, Vsum = mom_tr_grn.Rnum, mom_tr_grn.Vsum
with np.errstate(divide="ignore", invalid="ignore"):
    Zpg = np.where(Vsum > 0, Rnum / np.sqrt(Vsum), np.nan)
zg = Zpg[keep]
zg = zg[np.isfinite(zg)]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(-6, 6, 80)
ax[0].hist(
    np.clip(zg, bins[0], bins[-1]),
    bins=bins,
    density=True,
    color="C0",
    alpha=0.6,
    label=f"z  (sd={zg.std():.2f}, |z|>3: {(np.abs(zg) > 3).mean():.3f}, n={len(zg):,})",
)
xs = np.linspace(-6, 6, 400)
ax[0].plot(xs, np.exp(-(xs**2) / 2) / np.sqrt(2 * np.pi), "k--", lw=1, label="N(0,1)")
ax[0].set_xlabel(r"per-(perturbation, gene) $z = R/\sqrt{V}$  (ctrl + own-TF KDs)")
ax[0].set_ylabel("density")
ax[0].set_title("gene-level z — relevant perturbations only")
ax[0].legend(fontsize=8)

q = np.linspace(0.001, 0.999, 400)
ax[1].plot(stats.norm.ppf(q), np.quantile(zg, q), ".", ms=3, color="C0")
lo, hi = stats.norm.ppf([0.001, 0.999])
ax[1].plot([lo, hi], [lo, hi], "k--", lw=1)
ax[1].set_xlabel("theoretical N(0,1) quantile")
ax[1].set_ylabel("empirical z quantile")
ax[1].set_title("QQ plot")
plt.tight_layout()
plt.show()

print(
    f"restricted: n={len(zg):,}  sd={zg.std():.2f}  min={zg.min():.1f}  max={zg.max():.1f}  "
    f"|z|>3: {(np.abs(zg) > 3).mean():.3f}  |z|>6: {(np.abs(zg) > 6).mean():.4f}"
)

### 3.2 Normalized TF scores (histogram)

Distribution of the **regulon-level** TF scores on train — the aggregated score
itself, one step up from the per-gene `z` of §5.1. `raw Z` is the studentized
regulon residual; `Z_emp` is its per-TF (median/MAD) normalized version (the
discovery score). Compared against `N(0,1)`, this is the "fat tails" view: if the
gene-level `z` was ~normal but this is not, the excess is produced by the
aggregation (cross-gene covariance + coherent shift), not the NB studentization.

In [ ]:
z_tr = tfp.tf_zscores(
    mom_tr_grn, W_signed=W_signed, Amask_tf=mask["Amask_tf"], recalibrate=True
)
Zraw = z_tr.Z.ravel()
Zraw = Zraw[np.isfinite(Zraw)]
Znorm = z_tr.Z_emp.ravel()
Znorm = Znorm[np.isfinite(Znorm)]

bins = np.linspace(-10, 10, 80)
_groups = [
    (Zraw, "raw Z (studentized)", CMAP["C1"]),
    (Znorm, "Z_emp (per-TF normalized)", CMAP["C0"]),
]
_labels = {
    name: (
        f"{name}  (sd={v.std():.1f}, |z|>3: {(np.abs(v) > 3).mean():.3f}, "
        f"max|z|={np.abs(v).max():.0f}, n={len(v):,})"
    )
    for v, name, color in _groups
}
_h = pd.concat(
    pd.DataFrame({"v": np.clip(v, bins[0], bins[-1]), "dist": _labels[name]})
    for v, name, color in _groups
)
_fill = {_labels[name]: color for v, name, color in _groups}
xs = np.linspace(-10, 10, 400)
_norm = pd.DataFrame({"x": xs, "d": np.exp(-(xs**2) / 2) / np.sqrt(2 * np.pi)})
p = (
    gg.ggplot()
    + gg.geom_histogram(
        gg.aes("v", y=gg.after_stat("density"), fill="dist"),
        data=_h,
        bins=80,
        alpha=0.45,
        position="identity",
        color="none",
    )
    + gg.geom_line(gg.aes("x", "d"), data=_norm, linetype="dashed", size=0.5)
    + gg.scale_fill_manual(values=_fill, name="")
    + gg.labs(
        x="TF regulon score (train)",
        y="density",
        subtitle="§5.2 — TF-level scores vs N(0,1): fat tails after aggregation",
    )
    + gg.theme(
        legend_position="bottom",
    )
)
show_save(p, "fig_5_2_regulon_hist", size=(3.6, 2.6))

### 3.3 Figure 2 — self-regulon sanity check (do this first)

Score each TF-KD on **its own regulon**. If the model captures TF→gene logic,
the raw `Z` sits near 0 (the model predicts the response from the knocked-down
TF level). ConstantMean ignores TF level → large `Z`. The gap *is* the GRN's
value, at the single-regulon level. NB: TF-KDs are in the fit set, so this is
representational adequacy, not generalization.

In [ ]:
SELF.keys()

In [ ]:
from scipy.stats import norm

_ecdf = []
_color = {}
for name in [
    "Linear GRN (TFRegression)",
    "GRN + fitness (TFRegressionFixedNuisance)",
    "ConstantMean",
]:
    v = np.sort(SELF[name])
    iqr = np.subtract(*np.percentile(v, [75, 25]))
    lab = (
        f"{name}  (sd={v.std():.1f}, IQR={iqr:.1f}, "
        f"|Z|>3: {(np.abs(v) > 3).mean():.2f}, n={len(v)})"
    )
    _ecdf.append(
        pd.DataFrame({"v": v, "frac": np.arange(1, len(v) + 1) / len(v), "model": lab})
    )
    _color[lab] = CMAP[COLORS[name]]
_ecdf = pd.concat(_ecdf)

hi = np.abs(np.concatenate(list(SELF.values()))).max()
xs = np.linspace(-hi, hi, 20000)
_cdf = pd.DataFrame({"x": xs, "cdf": norm.cdf(xs)})
p = (
    gg.ggplot()
    + gg.geom_step(gg.aes("v", "frac", color="model"), data=_ecdf, size=0.6)
    + gg.geom_line(gg.aes("x", "cdf"), data=_cdf, linetype="dashed", size=0.5)
    + gg.scale_x_continuous(trans="symlog", minor_breaks=[])
    + gg.scale_color_manual(values=_color, name="")
    + gg.labs(
        x="raw Z on (TF-KD, its own regulon)",
        y="cumulative fraction",
        subtitle="Figure 2 — self-regulon scores (ECDF, symlog x)",
    )
    + gg.theme(
        legend_position="bottom",
    )
)
show_save(p, "fig2_self_regulon_ecdf", size=(3.6, 2.4))

In [ ]:
# Diagnostic: why is self-regulon Z so far from N(0,1)?  Decompose
#   Z_model = sqrt(n) * Sbar / sd_model   (denominator assumes gene-independence + NB-only noise)
#   T_emp   = sqrt(n) * Sbar / sd_emp     (empirical across-cell sd of the projection; note §5.1)
# Then:
#   |Z|/|T| = sd_emp/sd_model = variance INFLATION (mechanism a: cross-gene covariance)
#   |T| or d_emp=Sbar/sd_emp = is the regulon mean GENUINELY shifted (mechanism b: coherent bias, x sqrt(n))
import jax


@jax.jit
def _mean_only(params, x_tf, lib):
    mean, _ = grn.apply({"params": params}, x_tf, lib)
    return mean


W_eff_ = np.asarray(params_grn["W"]) * mask["Amask_tf"]
Wsgn_ = np.sign(W_eff_)
tf_col = {t: j for j, t in enumerate(tf_genes)}
gidx = {g: i for i, g in enumerate(adata.var_names)}

Xtf, Yraw, lib = arrays_train
labels = train_pert_labels
self_tfs = [t for t in tf_genes if t in set(labels.tolist())]
acc = {t: [0.0, 0.0, 0] for t in self_tfs}  # sum, sumsq, n

B = 512
for s in range(0, len(labels), B):
    sl = slice(s, s + B)
    mean = np.asarray(
        _mean_only(params_grn, jnp.asarray(Xtf[sl]), jnp.asarray(lib[sl]))
    )
    R = np.asarray(Yraw[sl]) - mean
    lab = labels[sl]
    for t in self_tfs:
        m = lab == t
        if not m.any():
            continue
        j = tf_col[t]
        proj = R[m] @ Wsgn_[:, j]
        gi = gidx.get(t, -1)
        if gi >= 0:  # match mask_perturbed_gene: drop the KD'd gene
            proj = proj - R[m][:, gi] * Wsgn_[gi, j]
        acc[t][0] += float(proj.sum())
        acc[t][1] += float((proj**2).sum())
        acc[t][2] += int(m.sum())

rows = []
for t in self_tfs:
    ssum, sq, n = acc[t]
    if n < 5:
        continue
    sbar = ssum / n
    var_emp = (sq - n * sbar**2) / (n - 1)
    if var_emp <= 0:
        continue
    i = np.where(mom_tr_grn.perts == t)[0]
    rows.append(
        {
            "TF": t,
            "n": n,
            "k": int(mask["Amask_tf"][:, tf_col[t]].sum()),
            "Sbar": sbar,
            "d_emp": sbar / np.sqrt(var_emp),  # per-cell effect size (NO sqrt-n)
            "T_emp": np.sqrt(n) * sbar / np.sqrt(var_emp),  # empirical t-stat
            "Z_model": float(z_tr_grn.Z[i[0], tf_col[t]]) if len(i) else np.nan,
        }
    )
diag = pd.DataFrame(rows)
diag["inflation_Z_over_T"] = diag["Z_model"].abs() / diag["T_emp"].abs()

print(f"self-regulon TFs analysed: {len(diag)}")
print(f"median |Z_model| : {diag['Z_model'].abs().median():.1f}")
print(
    f"median |T_emp|   : {diag['T_emp'].abs().median():.1f}   "
    f"(if still >>1 -> coherent mean bias, amplified by sqrt(n) -> a SPECIFICITY problem)"
)
print(
    f"median |d_emp|   : {diag['d_emp'].abs().median():.3f}   "
    f"(per-cell effect size; small => the huge T is pure sqrt(n) amplification)"
)
print(
    f"median inflation |Z|/|T| (= sd_emp/sd_model) : {diag['inflation_Z_over_T'].median():.1f}   "
    f"(>>1 -> denominator/cross-gene covariance is the culprit; the §5.1 t-test fixes THIS part)"
)

# (a) regulon size k (log x) vs variance inflation |Z|/|T|.
p = (
    gg.ggplot(diag, gg.aes("k", "inflation_Z_over_T"))
    + gg.geom_point(size=1.0, alpha=0.6, color=CMAP["C0"])
    + gg.scale_x_continuous(trans="log10")
    + gg.labs(
        x="regulon size k",
        y="|Z|/|T| (variance inflation)",
        subtitle="(a) covariance inflation grows with k?",
    )
)
show_save(p, "fig_5_3_inflation_vs_k", scatter=True, size=(3.2, 2.4))

# (b) sqrt(n_p) vs |T_emp| — coherent-bias amplification.
_diagb = diag.assign(sqrt_n=np.sqrt(diag["n"]), abs_T=diag["T_emp"].abs())
p = (
    gg.ggplot(_diagb, gg.aes("sqrt_n", "abs_T"))
    + gg.geom_point(size=1.0, alpha=0.6, color=CMAP["C0"])
    + gg.labs(
        x="√n_p",
        y="|T_emp|",
        subtitle="(b) does |T| scale with sqrt(n)? (coherent bias)",
    )
)
show_save(p, "fig_5_3_T_vs_sqrtn", scatter=True, size=(3.2, 2.4))

diag.reindex(diag["Z_model"].abs().sort_values(ascending=False).index).head(20)

## 4. Assemble `df_tf_enzymes` (all models)

In [ ]:
def long_frame(score_mat, name):
    return pd.DataFrame(
        {
            "perturbation": np.repeat(enzymatic_perts, n_tfs),
            "TF": np.tile(tf_genes, len(enzymatic_perts)),
            "score": np.asarray(score_mat).ravel(),
            "gt_sop": Y.ravel(),
            "coverage": cov.ravel(),
            "model": name,
        }
    )


df_tf_enzymes = pd.concat(
    [long_frame(s, n) for n, s in MODELS.items()], ignore_index=True
)
df_tf_enzymes.head()

### 4.1 Is the discovery ranking confounded by cell count `n_p`?

The diagnostic in §5 showed the studentized score behaves like `√n_p · (effect)`,
so a perturbation with more cells gets a larger score *for the same biology*.
PR / precision@K rank over (perturbation, TF) cells, so if `|Z_emp|` tracks `n_p`
the ranking is partly a cell-count ranking. `Z_emp` recalibrates per-TF (columns)
but `n_p` varies per-perturbation (rows), so it does **not** remove this.

Contrast `Z`-based scores (expect **+** corr, significance grows with n) against
DE mean `|LFC|` (an effect size; expect little **+** corr — if anything **−**,
since small-`n_p` pseudobulk means are noisier and inflate `|LFC|`).

In [ ]:
from scipy.stats import spearmanr

n_p_enz = np.asarray(mom_val_grn.counts)[enzymatic_idx]  # cells / enzyme perturbation
np_grid = np.broadcast_to(n_p_enz[:, None], (len(enzymatic_perts), n_tfs))

print(
    "Spearman(discovery score, n_p) over covered & finite (perturbation, TF) cells:\n"
)
for name, s in MODELS.items():
    sc = np.asarray(s)
    m = cov & np.isfinite(sc)
    rho, _ = spearmanr(np_grid[m], sc[m])
    print(f"  {name:30s}  rho = {rho:+.3f}   (n = {int(m.sum()):,})")

# does the TOP of the GRN ranking over-represent high-n_p perturbations?
sc = np.asarray(MODELS["Linear GRN (TFRegression)"])
m = cov & np.isfinite(sc)
flat_np, flat_sc = np_grid[m], sc[m]
order = np.argsort(-flat_sc)
print(
    f"\nmedian n_p:  all covered = {np.median(flat_np):.0f}   "
    f"top-50 = {np.median(flat_np[order[:50]]):.0f}   "
    f"top-200 = {np.median(flat_np[order[:200]]):.0f}"
)

# two-model scatter as a clean facet_wrap panel (log-log).
_frames = []
for name in ["Linear GRN (TFRegression)", "DE (mean |LFC|, raw)"]:
    sc = np.asarray(MODELS[name])
    m = cov & np.isfinite(sc)
    rho, _ = spearmanr(np_grid[m], sc[m])
    _frames.append(
        pd.DataFrame(
            {
                "n_p": np_grid[m],
                "score": sc[m],
                "model": f"{name}\nSpearman(score, n_p) = {rho:+.2f}",
            }
        )
    )
_scat = pd.concat(_frames)
p = (
    gg.ggplot(_scat, gg.aes("n_p", "score"))
    + gg.geom_point(size=0.4, alpha=0.2, color=CMAP["C0"])
    + gg.scale_x_continuous(trans="log10")
    + gg.scale_y_continuous(trans="log10")
    + gg.facet_wrap("model")
    + gg.labs(x="n_p (cells per perturbation)", y="discovery score")
)
show_save(p, "fig_6b_np_confound", scatter=True, size=(4.4, 2.2))

## 5. Discovery evaluation & interpretation

### 5.1 Figure 1 — PR curves (substrate∪product)

In [ ]:
rows = []
_curves = []
_color = {}
for name, s in MODELS.items():
    pr = tfp.pr_curve(s, Y, coverage=cov)
    lift = pr.auprc / max(pr.prevalence, 1e-12)
    lab = f"{name}  (AUPRC={pr.auprc:.3f}, lift={lift:.1f}x)"
    _curves.append(
        pd.DataFrame({"recall": pr.recall, "precision": pr.precision, "model": lab})
    )
    _color[lab] = CMAP[COLORS[name]]
    rows.append(
        {
            "model": name,
            "AUPRC": pr.auprc,
            "prevalence": pr.prevalence,
            "lift": lift,
            "n_pos": pr.n_positives,
            "n_total": pr.n_total,
        }
    )
_curves = pd.concat(_curves)

p = (
    gg.ggplot(_curves, gg.aes("recall", "precision", color="model"))
    + gg.geom_line(size=0.6)
    + gg.geom_hline(
        yintercept=rows[0]["prevalence"], color="black", linetype="dotted", size=0.5
    )
    + gg.scale_x_continuous(trans="log10")
    + gg.scale_color_manual(values=_color, name="")
    + gg.labs(
        x="recall",
        y="precision",
        subtitle="Figure 1 — |score| vs metabolic GT (substrate∪product)",
    )
)
show_save(p, "fig1_pr_curve", size=(5.0, 2.0))
pd.DataFrame(rows)

### 5.2 Figure 1b — precision@K & top-50 discoveries

In [ ]:
def precision_at_ks(score_mat, ks):
    s = np.asarray(score_mat).ravel()[cov.ravel()]
    y = Y.ravel()[cov.ravel()].astype(float)
    finite = np.isfinite(s)
    s, y = s[finite], y[finite]
    y = y[np.argsort(-s, kind="stable")]
    return {k: y[:k].sum() / k for k in ks}


prec = {name: precision_at_ks(s, PRECISION_KS) for name, s in MODELS.items()}
prevalence = Y[cov].mean()

_bar = pd.DataFrame(
    [
        {"K": f"K={k}", "model": name, "precision": prec[name][k]}
        for name in MODELS
        for k in PRECISION_KS
    ]
)
_bar["K"] = pd.Categorical(
    _bar["K"], categories=[f"K={k}" for k in PRECISION_KS], ordered=True
)
_bar["model"] = pd.Categorical(_bar["model"], categories=list(MODELS), ordered=True)
_color = {name: CMAP[COLORS[name]] for name in MODELS}
p = (
    gg.ggplot(_bar, gg.aes("K", "precision", fill="model"))
    + gg.geom_col(position="dodge")
    + gg.geom_hline(yintercept=prevalence, color="black", linetype="dotted", size=0.5)
    + gg.scale_fill_manual(values=_color, name="")
    + gg.labs(
        x="",
        y="precision@K",
        subtitle="Figure 1b — precision@K (substrate∪product)",
    )
)
show_save(p, "fig1b_precision_at_k", size=(3.6, 2.4))
pd.DataFrame(prec).T

In [ ]:
df_tf_enzymes["model"].unique()

In [ ]:
import json

grn_long = (
    df_tf_enzymes.query(
        # "model == 'Linear GRN (TFRegression)' and coverage"
        "model == 'GRN + fitness (TFRegressionFixedNuisance)' and coverage"
    )
    .loc[lambda d: np.isfinite(d["score"])]
    .sort_values("score", ascending=False)
)
grn_long["rank_in_TF"] = grn_long.groupby("TF").cumcount()

# cells per enzyme perturbation, to read alongside the score (it is an effect
# size, not a significance — small n_p hits may be under-powered / noisy)
n_p_enz = np.asarray(mom_val_grn.counts)[enzymatic_idx]
np_map = dict(zip(enzymatic_perts, n_p_enz))
grn_long = grn_long.assign(n_p=grn_long["perturbation"].map(np_map))

# substrate∪product positives available per TF (column) and per perturbation
# (row), over the covered candidate set — context for each hit: a discovery
# whose TF or enzyme has many positives is less surprising than a lone one.
n_pos_tf = grn_long.query("score >= 3.0").groupby("TF").size().astype(int)
n_pos_pert = grn_long.query("score >= 3.0").groupby("perturbation").size().astype(int)

# readable annotations (cf. tf_metabolic_pr.ipynb §4b):
#  - enzyme_sop : the perturbed enzyme's substrate∪product metabolites (iML1515)
#  - effector   : the TF's known effector metabolite(s) (sensor table), if any
with open(MODEL_JSON) as f:
    _bigg_name = {}
    for _m in json.load(f)["metabolites"]:
        _bigg_name.setdefault(_m["id"].rsplit("_", 1)[0], _m.get("name"))
_names = lambda ids: ", ".join(sorted({_bigg_name.get(b, b) for b in ids}))
_sop = lambda p: (
    _names(mm.enzyme_substrates(p) | mm.enzyme_products(p)) if mm.has_enzyme(p) else ""
)
_eff = (
    sensor[sensor["resolved"]]
    .assign(TF=lambda d: d["Transcription factor"].str.lower())
    .groupby("TF")["effector_name"]
    .agg(lambda s: ", ".join(sorted(set(s))))
)

# attach annotations to grn_long (so downstream slices inherit them)
grn_long["enzyme_sop"] = grn_long["perturbation"].map(_sop)
grn_long["effector"] = grn_long["TF"].map(_eff).fillna("")
grn_long["n_pos_TF"] = grn_long["TF"].map(n_pos_tf)
grn_long["n_pos_pert"] = grn_long["perturbation"].map(n_pos_pert)

In [ ]:
top50 = grn_long.head(50).copy()
print(
    f"top-50 GRN discoveries: {int(top50['gt_sop'].sum())}/50 are substrate∪product "
    f"positives (prevalence {prevalence:.3f})"
)
cols = [
    "perturbation",
    "TF",
    "score",
    "n_p",
    "n_pos_TF",
    "n_pos_pert",
    "enzyme_sop",
    "effector",
    "gt_sop",
]
top50[cols].reset_index(drop=True).style.set_properties(
    **{"white-space": "normal", "text-align": "left"}
)

### 5.3 Rank of the first hit, per TF

For every TF with coverage, rank its enzyme-KD candidates by `score` (descending)
*within that TF*, then record the rank of that TF's first substrate∪product
positive (`gt_sop`). rank 1 => the TF's single top-scoring knockdown is already a
true metabolic connection; a large rank => the method surfaces many
non-connected enzymes before the first real one. Unlike the global precision@K /
PR view, this is a per-TF retrieval diagnostic — it does not let one well-behaved
TF's many high scores mask another TF whose first hit is buried.

In [ ]:
# `grn_long` is GRN-only, coverage+finite, already sorted by score desc, so a
# per-TF cumcount gives the within-TF rank by score.
per_tf = grn_long.assign(rank_in_tf=lambda d: d.groupby("TF").cumcount() + 1)

first_hit = (
    per_tf.query("gt_sop")
    .groupby("TF", as_index=False)
    .first()  # first == highest-scoring positive for that TF
    .rename(
        columns={
            "rank_in_tf": "first_hit_rank",
            "perturbation": "first_hit_pert",
            "score": "first_hit_score",
        }
    )[["TF", "first_hit_rank", "first_hit_pert", "first_hit_score"]]
)

rank_first_hit = (
    per_tf.groupby("TF")
    .agg(n_candidates=("gt_sop", "size"), n_positives=("gt_sop", "sum"))
    .join(first_hit.set_index("TF"))
    .reset_index()
)
rank_first_hit["n_positives"] = rank_first_hit["n_positives"].astype(int)
rank_first_hit["first_hit_rank"] = rank_first_hit["first_hit_rank"].astype("Int64")
rank_first_hit["passes_thresh"] = rank_first_hit["first_hit_score"] >= THRESH
rank_first_hit = rank_first_hit.sort_values(
    ["first_hit_rank", "n_candidates"], na_position="last"
).reset_index(drop=True)

n_cov = len(rank_first_hit)
n_pos = int((rank_first_hit["n_positives"] > 0).sum())
n_rank1 = int((rank_first_hit["first_hit_rank"] == 1).sum())
print(
    f"{n_cov} TFs with coverage; {n_pos} have >=1 substrate\u222aproduct positive; "
    f"of those, {n_rank1} rank their first hit at position 1 "
    f"(median first-hit rank = {rank_first_hit['first_hit_rank'].median()})."
)
rank_first_hit

In [ ]:
# ECDF of the within-TF rank of the first hit (only TFs that have a positive).
_d = rank_first_hit.dropna(subset=["first_hit_rank"]).copy()
_d["first_hit_rank"] = _d["first_hit_rank"].astype(int)
p = (
    gg.ggplot(_d, gg.aes("first_hit_rank"))
    + gg.stat_ecdf(size=0.7)
    + gg.geom_vline(xintercept=1, color="black", linetype="dotted", size=0.5)
    + gg.scale_x_continuous(trans="log10")
    + gg.labs(
        x="rank of first hit within TF (log scale)",
        y="fraction of TFs (cumulative)",
        subtitle="Figure 1c \u2014 how deep is each TF's first metabolic hit?",
    )
)
show_save(p, "fig1c_first_hit_rank", size=(4.0, 2.4))

### 5.4 Interpretation of the GRN hits

Curated hits = top-1 enzyme KD per TF among substrate∪product positives with
`|Z_emp| >= THRESH`. This is the quantitative backbone of the slide-6 table and
the input to Figure 3.

In [ ]:
resolved = sensor[sensor["resolved"]].assign(
    TF=lambda d: d["Transcription factor"].str.lower()
)
eff_by_tf = resolved.groupby("TF").agg(
    effector_names=("effector_name", lambda s: sorted(set(s))),
    effector_bigg=("bigg_id", lambda s: sorted(set(map(str, s)))),
)

with open(MODEL_JSON) as f:
    _mets = json.load(f)["metabolites"]
bigg_name = {}
for m in _mets:
    bigg_name.setdefault(m["id"].rsplit("_", 1)[0], m.get("name"))
metabolite_names = lambda ids: ", ".join(sorted({bigg_name.get(b, b) for b in ids}))

pert_sets = {
    p: (mm.enzyme_substrates(p) | mm.enzyme_products(p)) for p in enzymatic_perts
}

In [ ]:
hits = (
    grn_long.query("gt_sop")
    .groupby("TF", as_index=False)
    .first()  # grn_long is already sorted by score desc
    # .query("score >= @THRESH")
    .sort_values("score", ascending=False)
)
hits["effector_names"] = hits["TF"].map(
    lambda t: (
        ", ".join(eff_by_tf.loc[t, "effector_names"]) if t in eff_by_tf.index else ""
    )
)
hits["enzyme_sop"] = hits["perturbation"].map(
    lambda p: metabolite_names(pert_sets.get(p, set()))
)
hits["n_pos_TF"] = hits["TF"].map(n_pos_tf)
hits["n_pos_pert"] = hits["perturbation"].map(n_pos_pert)
hits["n_edges"] = tf_info.set_index("TF").loc[hits["TF"], "n_edges"].values
print(f"{len(hits)} curated hits (top-1 enzyme per TF")
hits = (
    hits[
        [
            "perturbation",
            "TF",
            "effector_names",
            "enzyme_sop",
            "score",
            "n_pos_TF",
            "n_pos_pert",
            "n_edges",
            "rank_in_TF",
        ]
    ]
    .sort_values("rank_in_TF")
    .reset_index(drop=True)
)
hits.to_csv("top_hits.csv", index=False)
hits.head(50)

In [ ]:
TF_NAME = "argr"

In [ ]:
(
    grn_long.loc[lambda x: x["TF"] == TF_NAME]
    .assign(
        enzyme_sop=lambda x: x["perturbation"].map(
            lambda p: metabolite_names(pert_sets.get(p, set()))
        )
    )
    .assign(
        effector_names=lambda x: x["TF"].map(
            lambda t: (
                ", ".join(eff_by_tf.loc[t, "effector_names"])
                if t in eff_by_tf.index
                else ""
            )
        )
    )
    # .iloc[3]
    # .loc["enzyme_sop"]
    .head(25)
    .reset_index(drop=True)
    .loc[
        :,
        [
            "perturbation",
            "TF",
            "score",
            "enzyme_sop",
            "effector_names",
        ],
    ]
    .style.set_properties(**{"white-space": "normal", "text-align": "left"})
)

In [ ]:
tf_name = TF_NAME
targets = ref_db.query(f"regulator_gene == '{tf_name}'")
cross_table = ref_db.loc[ref_db["target_gene"].isin(targets["target_gene"])].loc[
    lambda x: x["regulator_gene"] != tf_name
]
tf_of_targets = sorted(set(cross_table["regulator_gene"]) - {tf_name})
print(
    f"{len(targets)} known targets of {tf_name}:",
    ", ".join(sorted(set(targets["target_gene"]))),
)
print("these targets are also regulated by:", ", ".join(tf_of_targets))
display(cross_table.groupby("target_gene")["regulator_gene"].agg(list).reset_index())

## 6. Figure 3 — does the inferred effector actually shift?

Per-example validation: for each curated hit, look up the TF's effector
metabolite in the Rapp 2026 metabolome (ln FC vs WT, same KD). Effectors not
measured by Rapp (e.g. Zn²⁺) drop out — this confirms interpretation where the
data exist, it is not a global statistic.

In [ ]:
rapp

In [ ]:
eff_by_tf

In [ ]:
rapp = pd.read_csv(RAPP_LFC_CSV, index_col=0)
rapp.index = rapp.index.str.lower()

recs = []
for _, r in hits.iterrows():
    p, t = r["perturbation"], r["TF"]
    if p not in rapp.index:
        continue
    known_effectors_bigg = (
        eff_by_tf.loc[t, "effector_bigg"] if t in eff_by_tf.index else []
    )
    for b in known_effectors_bigg:
        if b in rapp.columns and np.isfinite(rapp.loc[p, b]):
            recs.append(
                {
                    "perturbation": p,
                    "TF": t,
                    "effector": bigg_name.get(b, b),
                    "bigg": b,
                    "lnFC": float(rapp.loc[p, b]),
                    "score": r["score"],
                }
            )
meas = pd.DataFrame(recs)
print(f"{len(meas)} (hit, effector) pairs with a measured metabolite ln-FC")

if len(meas):
    meas = meas.sort_values("lnFC").reset_index(drop=True)
    meas["label"] = (
        meas["perturbation"] + " KD → " + meas["TF"] + " (" + meas["effector"] + ")"
    )
    meas["label"] = pd.Categorical(
        meas["label"], categories=meas["label"].tolist(), ordered=True
    )
    meas["sign"] = np.where(meas["lnFC"] < 0, "down", "up")
    p = (
        gg.ggplot(meas, gg.aes("label", "lnFC", fill="sign"))
        + gg.geom_col()
        + gg.geom_hline(yintercept=0, color="black", size=0.4)
        + gg.coord_flip()
        + gg.scale_fill_manual(
            values={"down": CMAP["C3"], "up": CMAP["C0"]}, guide=None
        )
        + gg.labs(
            x="",
            y="measured effector ln(FC vs WT)  [Rapp 2026]",
            subtitle="Figure 3 — inferred effector vs measured metabolite shift",
        )
        + gg.theme(figure_size=(3.6, 5.0))
    )
    show_save(p, "fig3_effector_shift", size=(5.0, 5.0))
# meas